**Eye-Tracking & Neurocognitive Intelligence | Reading Comprehension & Neurocognitive Modelling | Presented by: Reza Saadatyar (2026)**

**03 | Classical ML and Text-Based Baselines**

![Classical ML and Text-Based Baselines — scientific workflow](../../images/03_classical_ml_and_text_baselines.svg)

**Objective:** Compare gaze-only, text-only and combined classical predictors, including a training-fitted lexical TF-IDF comparator.

**Dataset:** Trial-level feature table, fixed participant assignments and local paragraph-text table from Notebook 02.

**Scope:** Predict the recorded correctness of a paragraph comprehension response from information available by the end of ordinary reading. No post-answer features, clinical inference or cognitive diagnosis.



**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess
import os
import sys

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
os.chdir(repo_path)
study_path = repo_path / "reading_comprehension_modelling"
if not (study_path / "code" / "src" / "onestop_io.py").is_file():
    raise FileNotFoundError("Update this checkout using git pull --ff-only origin main before running.")

# --------------------------- 3. Study Dependencies ---------------------------
# Existing compatible Colab packages are reused.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(study_path / "requirements.txt")], check=True)


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("reading_comprehension_modelling", sync_mode="off",
                   module_refresh="off")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

# ----------------- 4. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from onestop_io import validate_splits, validate_trial_index
from reading_models import classical_predictions, CLASSICAL
from reading_evaluation import metric

# ------------------------ 5. Output and Source Paths -------------------------
study = root / "reading_comprehension_modelling"
out_dir = study / "output"
out_dir.mkdir(parents=True, exist_ok=True)
raw_dir = root / "data" / "raw" / "onestop" / "ordinary"
processed = root / "data" / "processed" / "onestop"
processed.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
           label_size=11.5, label_weight="normal", tick_size=10, legend_size=10,
           figure_dpi=100, figure_size=(10, 5))


**02. Load Matched Trial Features and Participant Roles**

Read the fixed trial table and split assignments. The local paragraph table contains text for the lexical comparator; it must match the indexed trials. The model allowlists exclude all answer and identity fields.

In [ ]:
# ============ 02. Load Matched Trial Features and Participant Roles =============

identity = ["trial_id","participant_id","trial_index","article_batch",
            "article_key","paragraph_key","question_key","difficulty_level"]
table = pd.read_csv(out_dir/"02_trial_features.csv",dtype={c:str for c in identity})
index = pd.read_csv(out_dir/"01_trial_index.csv",dtype=str)
index["is_correct"] = index.is_correct.astype(int)
validate_trial_index(index,table)
splits = pd.read_csv(out_dir/"01_participant_splits.csv",dtype=str)
paragraph_text = pd.read_csv(processed/"02_paragraph_text.csv",dtype=str)
validate_splits(table,splits)
print("Classical candidates:",CLASSICAL)


**03. Fit Classical Models and Lexical Baselines**

Compare a Training prevalence reference, Logistic Regression, Random Forest and XGBoost for gaze-only, text-only and combined scalar predictors. Also compare paragraph TF-IDF Logistic Regression with and without gaze features. Imputation, scaling, vocabulary and class weights are Training-only. The decision threshold is fixed at 0.5.

In [ ]:
# ================ 03. Fit Classical Models and Lexical Baselines ================

classical, importance = classical_predictions(table,splits,paragraph_text,seed=17)
print("Prediction rows:",len(classical))


**04. Examine Validation Evidence**

Only Validation labels are scored here. Test scores are saved without outcome columns; Notebook 05 uses the fixed candidates and Validation selection before calculating Test metrics. Permutation importance for scalar predictors uses Validation only and measures model dependence, not causal effects.

In [ ]:
# ======================= 04. Examine Validation Evidence ========================

v = classical[classical.role.eq("validation")].merge(
    table[["trial_id","is_correct"]],on="trial_id",validate="many_to_one")
validation = pd.DataFrame([dict(protocol=p,model=m,**metric(g.is_correct,g.p_correct))
                           for (p,m),g in v.groupby(["protocol","model"])])
display(validation.sort_values(["protocol","macro_f1"],ascending=[True,False]))
fig, ax = plt.subplots(figsize=(11,6))
validation.pivot(index="model",columns="protocol",values="macro_f1").plot.barh(ax=ax)
ax.set(xlabel="Validation Macro F1",ylabel="",title="Classical and lexical candidates")
fig.tight_layout()
save_figure(fig,out_dir/"03_validation_comparison.png")
plt.show()


**05. Save Classical Prediction Scores**

Export one score per candidate and eligible Validation or Test trial. The scalar-model permutation summaries are also saved. No Test-informed fitting or selection occurs in this notebook.

In [ ]:
# ===================== 05. Save Classical Prediction Scores =====================

classical.to_csv(out_dir/"03_classical_predictions.csv",index=False)
validation.to_csv(out_dir/"03_validation_metrics.csv",index=False)
importance.to_csv(out_dir/"03_validation_feature_importance.csv",index=False)
download_outputs_from_colab(out_dir,"03")
